 **12a - RFMQ Model: Weight Matrix Build & Registration**

 Model 4 in the BharatMart 360° pipeline. No training step — the weight
 matrix IS the model. Computes Q from slv_cart_events, builds the 3×4
 segment × priority matrix, logs to MLflow, registers to Unity Catalog.

**Paper:** Chen, Q. et al. (2025). Frontiers in Big Data, Vol. 8.
DOI: https://doi.org/10.3389/fdata.2025.1680669

In [0]:
# COMMAND ----------

import json
import mlflow
import mlflow.pyfunc
from mlflow import MlflowClient
from pyspark.sql import functions as F
from pyspark.sql.functions import col, avg, sum as spark_sum
from pyspark.sql.functions import count, round as spark_round
from pyspark.sql.functions import when, lit, desc
from itertools import chain
from pyspark.sql.functions import create_map

from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()
print("imports ok")

**Load cart events**

In [0]:
# Q is computed from cart_add actions — explicit quantity integer per event
cart = spark.table("bharatmart.silver.slv_cart_events")
print(f"cart_events rows: {cart.count():,}")
cart.printSchema()

**Filter cart_add only**

In [0]:
cart_add = cart.filter(col("action") == "cart_add")
cart_add = cart_add.filter(col("customer_id").isNotNull())
cart_add = cart_add.filter(col("quantity") > 0)
print(f"cart_add rows: {cart_add.count():,}")

**Items per session**

In [0]:
items_per_session = (
    cart_add
    .groupBy("customer_id", "session_id")
    .agg(spark_sum("quantity").alias("items_in_session"))
)

print(f"sessions: {items_per_session.count():,}")
items_per_session.show(5)

**Compute Q per customer**

In [0]:
# average items across sessions per customer = Q dimension
q_df = (
    items_per_session
    .groupBy("customer_id")
    .agg(
        spark_round(avg("items_in_session"), 2).alias("avg_qty_per_order"),
        count("session_id").alias("session_count")))

print(f"customers with Q: {q_df.count():,}")
q_df.show(5)

**Q distribution check**

In [0]:
# understand Q spread before joining to segments
q_df.select(
    F.min("avg_qty_per_order").alias("min_Q"),
    F.percentile_approx("avg_qty_per_order", 0.25).alias("p25_Q"),
    F.percentile_approx("avg_qty_per_order", 0.50).alias("median_Q"),
    F.percentile_approx("avg_qty_per_order", 0.75).alias("p75_Q"),
    F.max("avg_qty_per_order").alias("max_Q"),
    F.mean("avg_qty_per_order").alias("mean_Q")
).show()

Q is tightly distributed. median is 3.25 items per order session,
mean is 3.30, IQR runs from 2.85 to 3.69. the maximum of 14 is
plausible for a bulk buyer, no clipping needed.

Chen et al. (2025) introduced Q precisely to separate customers
who have the same R, F, M but different basket sizes. a customer
who buys 1 expensive item per session behaves differently from one
who buys 10 cheap items. on BharatMart the separation is moderate,
IQR of less than 1 item, which means Q adds genuine signal without
overwhelming the RFM dimensions already computed in Model 2.

91,087 out of 91,634 total customers have a Q value, 99.4% coverage.
the 547 customers with no cart signal at all will receive the median
Q of 3.25 as a fallback when we join to the RFM segments.

**Q distribution chart**

In [0]:
import matplotlib.pyplot as plt

q_vals = q_df.select("avg_qty_per_order").toPandas()

plt.figure(figsize=(10, 5))
plt.hist(q_vals["avg_qty_per_order"], bins=50, color="#1B3A6B", edgecolor="white")
plt.axvline(3.25, color="red", linestyle="--", label="median Q = 3.25")
plt.axvline(3.30, color="pink", linestyle="--", label="mean Q = 3.30")
plt.title("Q Distribution — Average Items Per Order Session")
plt.xlabel("avg_qty_per_order")
plt.ylabel("customers")
plt.legend()
plt.tight_layout()
plt.show()

the distribution is approximately normal with a slight right skew.
the peak sits between 3.0 and 3.5 items per session, and the median
and mean are almost identical at 3.25 and 3.30. this confirms Q is
well-behaved with no transformation needed before using it in the
weight matrix.

the long right tail stretches to 14 but very few customers sit there.
beyond 6 items per session the counts drop to near zero. these are
the bulk buyers Chen et al. (2025) describe as having distinct
recommendation needs compared to single-item buyers.

median and mean being this close means the fallback value of 3.25
assigned to the 2,704 customers with no cart signal is a genuinely
neutral assumption, not an overestimate or underestimate.

**Load RFM segments**

In [0]:
rfm = spark.table("bharatmart.ml.gld_rfm_segments").select(
    "customer_id", "rfm_segment", "retention_priority",
    "recency_days", "total_orders", "total_spent"
)

print(f"RFM customers: {rfm.count():,}")
rfm.show(5)

**Join Q to RFM**

In [0]:
rfmq = rfm.join(q_df, "customer_id", "left")

print(f"after join: {rfmq.count():,}")
print(f"null Q count: {rfmq.filter(col('avg_qty_per_order').isNull()).count():,}")

93,784 customers in gld_rfm_segments. after joining Q, all 93,784
rows are preserved. 2,704 customers have no cart_add signal at all
and will receive the median Q of 3.25 as fallback.

2,704 is 2.9% of the base, consistent with the cold start rate we
found in Model 3. these are likely the same low-activity customers
who had fewer than 3 orders. they have no basket size signal because
they barely interacted with the cart at all.

**Q by segment chart**

In [0]:
q_by_segment = (
    rfmq.groupBy("rfm_segment")
    .agg(spark_round(avg("avg_qty_per_order"), 2).alias("avg_Q"))
    .toPandas()
    .sort_values("avg_Q", ascending=False)
)

plt.figure(figsize=(8, 5))
plt.bar(q_by_segment["rfm_segment"], q_by_segment["avg_Q"], color="#1B3A6B", edgecolor="white")
plt.title("Average Q by RFM Segment")
plt.xlabel("segment")
plt.ylabel("avg items per order session")
plt.tight_layout()
plt.show()

print(q_by_segment)

Q is almost identical across all three segments. Champions 3.30,
Loyal Customers 3.30, At Risk 3.29. the difference is 0.01 items
per session, which is negligible.

this tells us that on BharatMart, basket size does not separate
high-value customers from disengaged ones. Champions buy the same
number of items per session as At Risk customers. what separates
them is how recently they bought, how often, and how much they
spent, not how many items they put in the cart each time.

Chen et al. (2025) found Q meaningful in their outdoor sports
cross-border dataset where bulk buyers formed a distinct group.
on BharatMart Q does not create segment separation on its own.
this is not a problem for the model. Q still flows into the RFMQ
weight matrix as a fourth dimension, but the weight matrix itself
is driven by rfm_segment and retention_priority which do show
meaningful separation. Q validates that the three segments are
genuinely similar in basket behaviour, which means the RFM
dimensions from Model 2 are doing the heavy lifting correctly.

**Fill null Q with median fallback**

In [0]:
# customers with no cart signal get median Q as neutral fallback
median_q = q_df.selectExpr(
    "percentile_approx(avg_qty_per_order, 0.5) as median_q"
).collect()[0]["median_q"]

rfmq = rfmq.withColumn(
    "avg_qty_per_order",
    when(col("avg_qty_per_order").isNull(), lit(float(median_q)))
    .otherwise(col("avg_qty_per_order"))
)

print(f"median Q fallback: {median_q}")
print(f"null Q after fill: {rfmq.filter(col('avg_qty_per_order').isNull()).count():,}")

2,704 customers who had no cart_add signal received Q = 3.25,
the population median. null Q count is now 0 across all 93,784
customers. the dataset is complete and ready for the weight matrix.

3.25 is a safe neutral assignment. as the Q by segment chart showed,
all three segments sit between 3.29 and 3.30, so assigning 3.25 to
unknown customers places them just below the population average
without pushing them into any extreme bucket.

In [0]:
rfmq_pdf = rfmq.toPandas()
rfmq_pdf["total_orders"] = rfmq_pdf["total_orders"].astype(float)
rfmq_pdf["total_spent"]  = rfmq_pdf["total_spent"].astype(float)

features = rfmq_pdf[["recency_days", "total_orders", "total_spent", "avg_qty_per_order"]].copy()

scaler = StandardScaler()
scaled = scaler.fit_transform(features)

rfmq_pdf["R_scaled"] = scaled[:, 0] * -1   # invert R — lower recency = better
rfmq_pdf["F_scaled"] = scaled[:, 1]
rfmq_pdf["M_scaled"] = scaled[:, 2]
rfmq_pdf["Q_scaled"] = scaled[:, 3]

print("Z-score standardized + R inverted")
print(f"R_scaled mean: {rfmq_pdf['R_scaled'].mean():.4f}, std: {rfmq_pdf['R_scaled'].std():.4f}")
print(f"F_scaled mean: {rfmq_pdf['F_scaled'].mean():.4f}, std: {rfmq_pdf['F_scaled'].std():.4f}")
print(f"M_scaled mean: {rfmq_pdf['M_scaled'].mean():.4f}, std: {rfmq_pdf['M_scaled'].std():.4f}")
print(f"Q_scaled mean: {rfmq_pdf['Q_scaled'].mean():.4f}, std: {rfmq_pdf['Q_scaled'].std():.4f}")

z-score standardization confirmed. all four means are 0.0000 and all
stds are 1.0000 which is exactly what StandardScaler should produce.
R_scaled is already inverted so a customer who purchased 2 days ago
has a large positive R_scaled and a dormant customer has a large
negative value. all four dimensions now point the same direction:
higher = better. this matches paper section 3.1.2 step 6.

In [0]:
scaled_cols = ["R_scaled", "F_scaled", "M_scaled", "Q_scaled"]
data = rfmq_pdf[scaled_cols].copy()

In [0]:
for c in scaled_cols:
    shift = abs(data[c].min()) + 0.0001
    data[c] = data[c] + shift

m = len(data)

proportions = data.div(data.sum(axis=0), axis=1)

In [0]:
k = 1.0 / np.log(m)
entropy = {}
for c in scaled_cols:
    p = proportions[c].values
    # avoid log(0) — where p is 0, contribution is 0
    log_p = np.where(p > 0, np.log(p), 0)
    e_j = -k * np.sum(p * log_p)
    entropy[c] = e_j

print("entropy per dimension:")
for c, e in entropy.items():
    print(f"  {c}: {e:.6f}")

In [0]:
d = {c: 1 - e for c, e in entropy.items()}  # divergence
d_sum = sum(d.values())

weights = {c: d[c] / d_sum for c in scaled_cols}

print(f"\nentropy weights (our data):")
print(f"  W_R = {weights['R_scaled']:.4f}")
print(f"  W_F = {weights['F_scaled']:.4f}")
print(f"  W_M = {weights['M_scaled']:.4f}")
print(f"  W_Q = {weights['Q_scaled']:.4f}")
print(f"  sum = {sum(weights.values()):.4f}")

print(f"\npaper weights (Chen et al. 2025):")
print(f"  W_R = 0.0790")
print(f"  W_F = 0.3900")
print(f"  W_M = 0.1454")
print(f"  W_Q = 0.3856")

entropy weights computed. M (monetary) dominates massively at 0.7741.
this is the opposite of the paper where F and Q dominated at 0.39 each.

the entropy values explain why. M has the lowest entropy (0.922) meaning
monetary spend varies the most across BharatMart customers some spend
Rs.69,207, others spend Rs.200. that spread contains a lot of information
so the method assigns it the highest weight.

Q has the second highest entropy (0.995) meaning quantity barely varies
at all we already saw this. Q by segment was Champions=3.30, Loyal=3.30,
At Risk=3.29. essentially all 93,784 customers buy 3 items per order.
when a dimension has almost no variation it has almost no information
content, so the entropy method correctly gives it near-zero weight (0.0494).

R is similar entropy 0.999747 gives it W_R=0.0025. recency has almost
no discriminating power on BharatMart data.

this is a genuine cross-dataset finding. the paper's outdoor sports
cross-border customers varied more on F and Q (different SKUs, different
basket sizes). BharatMart customers are more homogeneous on basket size
but vary enormously on spend. the entropy method picked that up automatically
without any manual input. this is exactly why the paper chose entropy
weights over manual assignment.

**Weight comparison chart**

In [0]:
labels = ["R (Recency)", "F (Frequency)", "M (Monetary)", "Q (Quantity)"]
our_w  = [weights["R_scaled"], weights["F_scaled"], weights["M_scaled"], weights["Q_scaled"]]
paper_w = [0.0790, 0.3900, 0.1454, 0.3856]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, our_w,   width, label="BharatMart (ours)", color="#1B3A6B", edgecolor="white")
bars2 = ax.bar(x + width/2, paper_w, width, label="Chen et al. 2025",  color="#9E9E9E", edgecolor="white")

ax.set_ylabel("Entropy Weight")
ax.set_title("RFMQ Entropy Weights — BharatMart vs Paper")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.set_ylim(0, 0.5)

for bar in bars1:
    ax.annotate(f"{bar.get_height():.3f}",
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 5), textcoords="offset points", ha="center", fontsize=9)
for bar in bars2:
    ax.annotate(f"{bar.get_height():.3f}",
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 5), textcoords="offset points", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

**RFMQ Composite Score Per Customer**

In [0]:
W_R = weights["R_scaled"]
W_F = weights["F_scaled"]
W_M = weights["M_scaled"]
W_Q = weights["Q_scaled"]

rfmq_pdf["rfmq_raw_score"] = (
    W_R * rfmq_pdf["R_scaled"] +
    W_F * rfmq_pdf["F_scaled"] +
    W_M * rfmq_pdf["M_scaled"] +
    W_Q * rfmq_pdf["Q_scaled"]
)

print(f"min  : {rfmq_pdf['rfmq_raw_score'].min():.4f}")
print(f"max  : {rfmq_pdf['rfmq_raw_score'].max():.4f}")
print(f"mean : {rfmq_pdf['rfmq_raw_score'].mean():.4f}")
print(f"std  : {rfmq_pdf['rfmq_raw_score'].std():.4f}")

raw score runs from -1.02 to 3.39 with mean 0. negative means below
average across all four dimensions, positive means above average.
the std of 0.81 shows reasonable spread. now normalize to [0.5, 1.5].

raw score is a continuous value. negative means the customer is below average across all four dimensions (low value, disengaged). positive means above average (high value, active).

we need to normalize this into a multiplier range so it works with ALS scores. the range 0.5 to 1.5 means:
- worst customers get 0.5x on their ALS scores (dampened)
- average customers get ~1.0x (neutral)
- best customers get 1.5x (boosted)

this is the same effect as the old 3x4 matrix (0.8 to 1.5) but now continuous and data-derived instead of discrete and handcrafted.

In [0]:
raw_min = rfmq_pdf["rfmq_raw_score"].min()
raw_max = rfmq_pdf["rfmq_raw_score"].max()

rfmq_pdf["rfmq_weight"] = (
    (rfmq_pdf["rfmq_raw_score"] - raw_min) / (raw_max - raw_min)
    * (1.5 - 0.5)
    + 0.5
)

print(f"rfmq_weight stats:")
print(f"  min    : {rfmq_pdf['rfmq_weight'].min():.4f}")
print(f"  max    : {rfmq_pdf['rfmq_weight'].max():.4f}")
print(f"  mean   : {rfmq_pdf['rfmq_weight'].mean():.4f}")
print(f"  median : {rfmq_pdf['rfmq_weight'].median():.4f}")

multiplier confirmed in range 0.5 to 1.5. median of 0.66 means most
customers sit in the lower half — expected given monetary spend is
heavily right-skewed. the small group of high-spend Champions pull
the max to 1.5 while the bulk of At Risk customers cluster near 0.5.

multiplier now runs from 0.5 to 1.5.

the mean and median tell us where most customers sit. if the median is around 1.0, half the customers get a neutral or positive boost and half get dampened. this is the right shape - Champions cluster near 1.5, At Risk customers cluster near 0.5.

**RFMQ Weight Distribution**

In [0]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(rfmq_pdf["rfmq_weight"], bins=50, color="#1B3A6B", edgecolor="white")
ax.axvline(1.0, color="red", linestyle="--", label="neutral (1.0x)")
ax.set_xlabel("RFMQ Weight Multiplier")
ax.set_ylabel("Customer Count")
ax.set_title("RFMQ Weight Distribution — Entropy-Derived Multiplier")
ax.legend()
plt.tight_layout()
plt.show()

The chart shows how customers are spread across a composite RFMQ score called the "weight multiplier." This single number combines each customer's Recency, Frequency, Monetary, and Quantity behavior into one value.

The first thing you notice is that most customers are clustered between 0.55 and 0.85, with the peak sitting around 0.62 to 0.65 where nearly 8,500 to 8,800 customers are grouped. This means the vast majority of your customer base scores below the neutral line of 1.0.

The red dashed line at 1.0 is important - it represents an "average" customer. Anything below it means the customer is less engaged or lower value than average. Since most of the distribution sits to the left of this line, it tells you that a large portion of your customers are low engagement and low spend.

On the right side of the chart, the scores stretch out to about 1.5, forming a long thin tail. These are your high-value or VIP customers. They are few in number but are likely responsible for a significant share of your revenue.

The overall shape of this distribution is a classic pattern in retail - a large base of low-to-moderate customers and a small but important group of high-value ones. In practical terms, this means your customer base needs two very different strategies: activation and nurturing for the large low-scoring majority, and retention and rewards for the small premium segment on the right tail.

**RFMQ Weight by Segment**

In [0]:
weight_by_seg = rfmq_pdf.groupby("rfm_segment")["rfmq_weight"].agg(["mean", "median", "min", "max"])
weight_by_seg = weight_by_seg.sort_values("mean", ascending=False)

print(weight_by_seg.round(4))

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(weight_by_seg.index, weight_by_seg["mean"], color="#1B3A6B", edgecolor="white")
ax.axhline(1.0, color="red", linestyle="--", label="neutral (1.0x)")
ax.set_xlabel("RFM Segment")
ax.set_ylabel("Mean RFMQ Weight")
ax.set_title("Mean RFMQ Weight by Segment")
ax.legend()
plt.tight_layout()
plt.show()

The chart shows the average RFMQ weight multiplier for three customer segments: Champions, Loyal Customers, and At Risk.

Champions have the highest mean weight at 0.8427, meaning they are the most engaged and highest value group overall. Their scores range from 0.52 at the lowest to a perfect 1.5 at the top, showing there is real variation even within this top segment. Still, even Champions sit below the neutral line of 1.0 on average, which tells you that truly "above average" customers are quite rare.

Loyal Customers sit in the middle with a mean of 0.6350. They are a reasonably consistent group, with scores tightly clustered between 0.57 and 0.76, and their mean and median are close to each other, suggesting a stable, predictable segment without many outliers.

At Risk customers have the lowest mean at 0.5693, barely above the minimum of 0.50. Their scores are compressed into a narrow band from 0.50 to 0.71, which means there is not much spread - they are uniformly disengaged rather than a mixed group.

The key takeaway is that all three segments fall below the 1.0 neutral line, confirming what the previous distribution chart showed - most customers across all segments are below average in composite RFMQ value. The gap between Champions and At Risk is meaningful but not dramatic, sitting at roughly 0.27 points apart. This suggests the segments are distinguishable but there is room to move customers up the ladder with the right interventions.

**RFMQ Weight by Retention Priority**

In [0]:
weight_by_pri = rfmq_pdf.groupby("retention_priority")["rfmq_weight"].agg(["mean", "median", "count"])

# order priorities logically
pri_order = ["Urgent", "High", "Medium", "Low"]
weight_by_pri = weight_by_pri.reindex(pri_order)

print(weight_by_pri.round(4))


Urgent customers have a lower mean weight (0.62) than High (0.78).
this makes sense - Urgent customers are Champions at churn risk,
meaning their recent activity has dropped. R is inverted so a
deteriorating recency pulls their score down even if M and F are
strong. the old matrix gave Urgent Champions a flat 1.5x regardless.
the entropy method correctly reflects that a churning Champion is
worth less right now than an active High-priority customer.

High has the most customers (68,515) and the highest mean weight -
these are active Champions and Loyal customers driving the bulk of
revenue.

**Set MLflow experiment**

In [0]:
mlflow.set_experiment("/Users/viresh.skyquest@gmail.com/RFMQ_Scoring")
print("experiment set")

**Log params, metrics, and artifacts**

In [0]:
entropy_weights = {
    "W_R": round(W_R, 6),
    "W_F": round(W_F, 6),
    "W_M": round(W_M, 6),
    "W_Q": round(W_Q, 6),
}

run = mlflow.start_run(run_name="rfmq_entropy_weights_v1")
run_id = run.info.run_id

mlflow.log_param("method", "entropy_weight_method")
mlflow.log_param("standardization", "z_score_r_inverted")
mlflow.log_param("multiplier_range", "0.5_to_1.5")
mlflow.log_param("W_R", entropy_weights["W_R"])
mlflow.log_param("W_F", entropy_weights["W_F"])
mlflow.log_param("W_M", entropy_weights["W_M"])
mlflow.log_param("W_Q", entropy_weights["W_Q"])

print(f"run started : {run_id}")
print(f"weights : {entropy_weights}")

In [0]:
mlflow.log_metric("total_customers", len(rfmq_pdf))
mlflow.log_metric("median_q", float(median_q))
mlflow.log_metric("q_null_fallback_count", 2704)
mlflow.log_metric("rfmq_weight_mean", round(rfmq_pdf["rfmq_weight"].mean(), 4))
mlflow.log_metric("rfmq_weight_median", round(rfmq_pdf["rfmq_weight"].median(), 4))
mlflow.log_metric("rfmq_weight_std", round(rfmq_pdf["rfmq_weight"].std(), 4))

print("metrics logged")

In [0]:
import pickle

rfmq_config = {
    "method": "entropy_weight_method",
    "weights": entropy_weights,
    "multiplier_range": [0.5, 1.5],
    "q_fallback_median": float(median_q),
    "standardization": "z_score",
    "r_inverted": True,
    "formula": "rfmq_score = W_R*R_scaled + W_F*F_scaled + W_M*M_scaled + W_Q*Q_scaled"
}

with tempfile.TemporaryDirectory() as tmp:
    config_path = os.path.join(tmp, "rfmq_weights.json")
    with open(config_path, "w") as f:
        json.dump(rfmq_config, f, indent=2)
    mlflow.log_artifact(config_path)

    scaler_path = os.path.join(tmp, "rfmq_scaler.pkl")
    with open(scaler_path, "wb") as f:
        pickle.dump(scaler, f)
    mlflow.log_artifact(scaler_path)

print("artifacts logged: rfmq_weights.json, rfmq_scaler.pkl")

**end run**

In [0]:
mlflow.end_run()
print(f"run complete : {run_id}")

In [0]:
rfmq_config_model = RFMQWeightModel(entropy_weights)

sample_input = pd.DataFrame({
    "rfm_segment": ["Champions"],
    "retention_priority": ["Urgent"],
    "als_score": [0.85]
})

sample_output = rfmq_config_model.predict(None, sample_input.copy())
signature = infer_signature(sample_input, sample_output)

with mlflow.start_run(run_name="rfmq_model_registration") as reg_run:
    mlflow.pyfunc.log_model(
        artifact_path="rfmq_model",
        python_model=rfmq_config_model,
        input_example=sample_input,
        signature=signature,
    )
    model_uri = f"runs:/{reg_run.info.run_id}/rfmq_model"

registered = mlflow.register_model(
    model_uri=model_uri,
    name="bharatmart.ml.bharatmart_rfmq_model"
)

print(f"registered version : {registered.version}")
print(f"name  : {registered.name}")

In [0]:
client.set_registered_model_alias(
    name="bharatmart.ml.bharatmart_rfmq_model",
    alias="Production",
    version=registered.version
)

print(f"alias set: Production -> version {registered.version}")

In [0]:
model_info = client.get_registered_model("bharatmart.ml.bharatmart_rfmq_model")
prod_version = client.get_model_version_by_alias(
    "bharatmart.ml.bharatmart_rfmq_model", "Production"
)

print(f"model    : {model_info.name}")
print(f"version  : {prod_version.version}")
print(f"alias    : Production")
print(f"run_id   : {prod_version.run_id}")